# ITS 2122 – Python for Data Science & AI

## Phase 01: Data Ingestion & Clinical Sanitation

### Project: Patient Risk Stratification – Vitality Health Network (VHN)

### Phase Overview

This phase focuses on preparing the raw clinical dataset for reliable
data analysis and patient risk stratification.

The main objectives of this phase are:

1. Load the clinical dataset into a Pandas DataFrame.
2. Perform an initial data quality and structure assessment.
3. Standardize non-standard missing values.
4. Analyze missing data and evaluate the `weight` feature.
5. Apply ID mappings to obtain meaningful descriptions.
6. Exclude encounters associated with expired patients.
7. Remove exact duplicate records while preserving valid patient encounters.

### Expected Outcome

A clean and structured encounter-level dataset that can be
used for the subsequent analytical phases.

In [3]:
# Step 1.1: Import the required libraries

import pandas as pd
import numpy as np

from pathlib import Path

# Display the installed library versions
print("Pandas Version:", pd.__version__)
print("NumPy Version:", np.__version__)

print("\nRequired libraries imported successfully.")

Pandas Version: 3.0.6
NumPy Version: 2.5.4

Required libraries imported successfully.


### 1.2 Loading the Clinical Dataset

The `diabetic_data.csv` file contains patient encounter information
collected from the hospital's electronic health record system.

The dataset is loaded into a Pandas DataFrame for further processing.

The original dataset uses the question mark (`?`) to represent
missing values in certain columns.

The `na_values` parameter converts these non-standard missing-value
markers into standard NumPy `NaN` values during data ingestion.

This ensures consistent missing-value identification throughout
the data cleaning process.

In [9]:
# Step 1.2: Define the dataset location

DATA_DIR = Path("data_files")

DATA_FILE = DATA_DIR / "diabetic_data.csv"

# Verify that the dataset file exists
if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Dataset file not found: {DATA_FILE.resolve()}"
    )

# Load the clinical dataset
df_raw = pd.read_csv(
    DATA_FILE,
    na_values=["?"],
    low_memory=False
)

# Display dataset dimensions
print("Clinical dataset loaded successfully!")

print("Number of Rows:", df_raw.shape[0])
print("Number of Columns:", df_raw.shape[1])

# Display the first five records
display(df_raw.head())

Clinical dataset loaded successfully!
Number of Rows: 101766
Number of Columns: 50


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


### 1.3 Initial Data Assessment

An initial assessment is performed to understand the structure,
features, data types, and statistical characteristics of the dataset.

The following Pandas methods are used:

1. `shape` – Identifies the number of rows and columns.
2. `columns` – Displays all available feature names.
3. `info()` – Provides information about data types and non-null values.
4. `describe()` – Generates descriptive statistics for dataset features.
5. `head()` – Displays the first five records.

Special attention is given to identifier columns because their
numerical values represent categories rather than measurable quantities.

This assessment helps identify potential data quality issues
before applying further cleaning operations.

In [10]:
# Step 1.3: Perform an initial assessment of the raw dataset

# Display the number of records and features
print("Dataset Dimensions:")
print(df_raw.shape)

# Display all column names
print("\nDataset Column Names:")
print(df_raw.columns.tolist())

# Display data types and non-null counts
print("\nDataset Information:")
df_raw.info()

# Generate descriptive statistics for all columns
print("\nDescriptive Statistical Summary:")
display(df_raw.describe(include="all").T)

# Display the first five records
print("\nFirst Five Records:")
display(df_raw.head())

Dataset Dimensions:
(101766, 50)

Dataset Column Names:
['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #  

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
encounter_id,101766.0,NaN,NaN,NaN,165201645.622978,102640295.983457,12522.0,84961194.0,152388987.0,230270887.5,443867222.0
patient_nbr,101766.0,NaN,NaN,NaN,54330400.694947,38696359.346534,135.0,23413221.0,45505143.0,87545949.75,189502619.0
race,99493,5,Caucasian,76099,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,101766,3,Female,54708,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,101766,10,[70-80),26068,NaN,NaN,NaN,NaN,NaN,NaN,NaN
weight,3197,9,[75-100),1336,NaN,NaN,NaN,NaN,NaN,NaN,NaN
admission_type_id,101766.0,NaN,NaN,NaN,2.024006,1.445403,1.0,1.0,1.0,3.0,8.0
discharge_disposition_id,101766.0,NaN,NaN,NaN,3.715642,5.280166,1.0,1.0,1.0,4.0,28.0
admission_source_id,101766.0,NaN,NaN,NaN,5.754437,4.064081,1.0,1.0,7.0,7.0,25.0
time_in_hospital,101766.0,NaN,NaN,NaN,4.395987,2.985108,1.0,2.0,4.0,6.0,14.0



First Five Records:


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


### 1.4 Loading the ID Mapping Dataset

The `IDs_mapping.csv` file contains descriptions for the
categorical identifier columns in the clinical dataset.

These mappings are required to interpret the numerical
identifiers correctly.

The following identifier columns are considered:

1. `admission_type_id`
2. `discharge_disposition_id`
3. `admission_source_id`

The mapping file is structured into separate sections.
Therefore, each section is extracted and stored as a
separate dictionary for further processing.

This approach improves the interpretability of the clinical
dataset and supports the identification of expired patients.

In [11]:
# Step 1.4: Load and extract ID mappings

import csv

# Define the mapping file location
MAPPING_FILE = DATA_DIR / "IDs_mapping.csv"

# Verify that the mapping file exists
if not MAPPING_FILE.exists():
    raise FileNotFoundError(
        f"Mapping file not found: {MAPPING_FILE.resolve()}"
    )

# Read the mapping file
with MAPPING_FILE.open(
    mode="r",
    encoding="utf-8-sig",
    newline=""
) as file:

    mapping_rows = list(csv.reader(file))

# Define the required mapping sections
required_sections = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id"
]

# Store extracted mappings
id_mappings = {}

# Extract each mapping section
for section in required_sections:

    mapping_dict = {}
    section_found = False

    for index, row in enumerate(mapping_rows):

        # Identify the beginning of the required section
        if row and row[0].strip().lower() == section:

            section_found = True

            # Read mapping records until the next section
            for mapping_row in mapping_rows[index + 1:]:

                if not mapping_row or not mapping_row[0].strip():
                    continue

                first_value = mapping_row[0].strip().lower()

                # Stop when another mapping section begins
                if first_value in required_sections:
                    break

                # Store the ID and its description
                if len(mapping_row) >= 2:
                    mapping_id = mapping_row[0].strip()
                    description = mapping_row[1].strip()

                    if mapping_id.isdigit():
                        mapping_dict[mapping_id] = description

            break

    if not section_found:
        raise ValueError(
            f"Mapping section '{section}' was not found."
        )

    id_mappings[section] = mapping_dict

# Display extracted mappings
for section, mapping_dict in id_mappings.items():

    print(f"\n{section}")
    print("-" * 50)

    for mapping_id, description in mapping_dict.items():
        print(f"{mapping_id}: {description}")

print("\nID mappings loaded successfully.")


admission_type_id
--------------------------------------------------
1: Emergency
2: Urgent
3: Elective
4: Newborn
5: Not Available
6: NULL
7: Trauma Center
8: Not Mapped

discharge_disposition_id
--------------------------------------------------
1: Discharged to home
2: Discharged/transferred to another short term hospital
3: Discharged/transferred to a Skilled Nursing Facility (SNF)
4: Discharged/transferred to ICF
5: Discharged/transferred to another type of inpatient care institution
6: Discharged/transferred to home with home health service
7: Left AMA
8: Discharged/transferred to home under care of Home IV provider
9: Admitted as an inpatient to this hospital
10: Neonate discharged to another hospital for neonatal aftercare
11: Expired
12: Still patient or expected to return for outpatient services
13: Hospice / home
14: Hospice / medical facility
15: Discharged/transferred within this institution to Medicare approved swing bed
16: Discharged/transferred/referred another instit

### 1.5 Applying ID Mappings

The numerical identifiers in the clinical dataset represent
categorical information rather than continuous numerical values.

The extracted ID mappings are used to create additional
description columns for better data interpretation.

The following description columns are generated:

- `admission_type_description`
- `discharge_disposition_description`
- `admission_source_description`

The original identifier columns are preserved to maintain
the integrity of the raw clinical data.

These descriptions will also help identify encounters
associated with expired patients.

In [12]:
# Step 1.5: Apply ID mappings to the clinical dataset

# Create a working copy of the original dataset
df_cleaned = df_raw.copy()

# Define the identifier columns and their description columns
mapping_columns = {
    "admission_type_id": "admission_type_description",
    "discharge_disposition_id": "discharge_disposition_description",
    "admission_source_id": "admission_source_description"
}

# Apply the extracted mappings
for id_column, description_column in mapping_columns.items():

    # Convert identifier values into consistent string format
    id_values = pd.to_numeric(
        df_cleaned[id_column],
        errors="coerce"
    ).astype("Int64").astype("string")

    # Add the corresponding descriptions
    df_cleaned[description_column] = id_values.map(
        id_mappings[id_column]
    )

# Display the mapped identifier columns
for id_column, description_column in mapping_columns.items():

    print(f"\n{id_column} and {description_column}")
    display(
        df_cleaned[
            [id_column, description_column]
        ].drop_duplicates().head(15)
    )

print("\nID descriptions applied successfully.")


admission_type_id and admission_type_description


,admission_type_id,admission_type_description
0,6,NULL
1,1,Emergency
5,2,Urgent
6,3,Elective
2043,4,Newborn
3089,5,Not Available
7789,8,Not Mapped
45829,7,Trauma Center



discharge_disposition_id and discharge_disposition_description


,discharge_disposition_id,discharge_disposition_description
0,25,Not Mapped
1,1,Discharged to home
9,3,Discharged/transferred to a Skilled Nursing Fa...
13,6,Discharged/transferred to home with home healt...
29,2,Discharged/transferred to another short term h...
31,5,Discharged/transferred to another type of inpa...
34,11,Expired
82,7,Left AMA
487,10,Neonate discharged to another hospital for neo...
972,4,Discharged/transferred to ICF



admission_source_id and admission_source_description


,admission_source_id,admission_source_description
0,1,Physician Referral
1,7,Emergency Room
5,2,Clinic Referral
8,4,Transfer from a hospital
335,5,Transfer from a Skilled Nursing Facility (SNF)
396,6,Transfer from another health care facility
469,20,Not Mapped
1014,3,HMO Referral
1039,17,NULL
1595,8,Court/Law Enforcement



ID descriptions applied successfully.


### 1.6 Missing Value Analysis

Missing values can affect the accuracy and reliability
of clinical data analysis.

Although the question mark (`?`) was converted into
standard missing values during data ingestion, a detailed
missing value assessment is required.

This section calculates:

1. Total missing values in each column.
2. Percentage of missing values in each column.
3. Columns containing missing data.

The results are arranged in descending order based on
the percentage of missing values.

This analysis helps identify columns that require
special handling before further data processing.

In [13]:
# Step 1.6: Analyze missing values in the dataset

# Calculate the total missing values in each column
missing_counts = df_cleaned.isnull().sum()

# Calculate the percentage of missing values in each column
missing_percentages = (
    df_cleaned.isnull().mean() * 100
)

# Create a missing value summary
missing_summary = pd.DataFrame({
    "Missing Count": missing_counts,
    "Missing Percentage (%)": missing_percentages
})

# Sort columns by missing percentage
missing_summary = missing_summary.sort_values(
    by="Missing Percentage (%)",
    ascending=False
)

# Display columns containing missing values
print("Missing Value Analysis:")

display(
    missing_summary[
        missing_summary["Missing Count"] > 0
    ]
)

# Display the total number of missing values
print(
    "\nTotal Missing Values:",
    df_cleaned.isnull().sum().sum()
)

Missing Value Analysis:


,Missing Count,Missing Percentage (%)
weight,98569,96.858479
max_glu_serum,96420,94.746772
A1Cresult,84748,83.277322
medical_specialty,49949,49.082208
payer_code,40256,39.557416
race,2273,2.233555
diag_3,1423,1.398306
diag_2,358,0.351787
diag_1,21,0.020636



Total Missing Values: 374017


### 1.7 Weight Feature Analysis

The `weight` column is examined to determine the extent
of missing information.

A missing value percentage greater than 90% indicates
that the feature contains insufficient information for
reliable analysis.

According to the project requirements, if more than 90%
of the values are missing, the entire `weight` column
must be removed.

This decision is documented as a Data Quality Limitation
because removing the feature results in the loss of
potentially useful clinical information.

If the missing percentage does not exceed 90%, the
column is retained for further consideration.

In [14]:
# Step 1.7: Analyze and handle missing values in the weight column

# Check whether the weight column exists
if "weight" in df_cleaned.columns:

    # Calculate the percentage of missing values
    weight_missing_percentage = (
        df_cleaned["weight"].isnull().mean() * 100
    )

    print("Weight Column Analysis")
    print("-" * 40)

    print(
        "Total Records:",
        len(df_cleaned)
    )

    print(
        "Missing Values:",
        df_cleaned["weight"].isnull().sum()
    )

    print(
        f"Missing Percentage: "
        f"{weight_missing_percentage:.2f}%"
    )

    # Apply the 90% missing value threshold
    if weight_missing_percentage > 90:

        df_cleaned.drop(
            columns=["weight"],
            inplace=True
        )

        print("\nData Quality Limitation:")
        print(
            "The weight column was removed because "
            "more than 90% of its values were missing."
        )

    else:

        print(
            "\nThe weight column is retained because "
            "its missing percentage does not exceed 90%."
        )

else:

    print(
        "The weight column does not exist in the dataset."
    )

Weight Column Analysis
----------------------------------------
Total Records: 101766
Missing Values: 98569
Missing Percentage: 96.86%

Data Quality Limitation:
The weight column was removed because more than 90% of its values were missing.
